# Workstream 5: Embedding EDA

track / user embeddings の null、zero vector、dim、norm distribution を確認し、dense retrieval と rerank feature の適用範囲を決めます。

主な既存成果物: `track_embedding_stats.csv`, `track_embedding_norms.csv`, `user_embedding_stats.csv`, `user_embedding_norms.csv`, `embedding_norms.png`.


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = Markdown = display = None

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 160)


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "EDA").exists():
            return candidate
    raise RuntimeError("Repository root was not found from the current working directory.")

ROOT = find_repo_root(Path.cwd())
EDA_DIR = ROOT / "EDA"
TABLE_DIR = EDA_DIR / "tables"
FIGURE_DIR = EDA_DIR / "figures"
SUMMARY_DIR = EDA_DIR / "summary"
EXPERIMENT_DIR = ROOT / "mcrs" / "experiments"
INFERENCE_DIR = ROOT / "exp" / "inference"

for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"TABLE_DIR={TABLE_DIR}")


In [ ]:
def read_table(name: str, **kwargs) -> pd.DataFrame:
    path = TABLE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def read_csv_path(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def show_df(df: pd.DataFrame, n: int = 20) -> None:
    if df.empty:
        print("empty dataframe")
        return
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n).to_string(index=False))


def show_image(name: str) -> None:
    path = FIGURE_DIR / name
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT)}")
        return
    if display is not None and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(path)


def save_table(df: pd.DataFrame, name: str) -> Path | None:
    if df.empty:
        print(f"skip empty table: {name}")
        return None
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print(f"saved: {path.relative_to(ROOT)} ({len(df):,} rows)")
    return path


def barplot(df: pd.DataFrame, *, x: str, y: str, hue: str | None = None, title: str = "", rotate: int = 0, figsize=(10, 4)) -> None:
    if df.empty:
        print("skip empty plot")
        return
    fig, ax = plt.subplots(figsize=figsize)
    if sns is not None:
        sns.barplot(data=df, x=x, y=y, hue=hue, ax=ax)
    else:
        df.plot(kind="bar", x=x, y=y, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=rotate)
    fig.tight_layout()
    plt.show()


## Load Embedding Tables


In [ ]:
track_stats = read_table("track_embedding_stats.csv")
track_norms = read_table("track_embedding_norms.csv")
user_stats = read_table("user_embedding_stats.csv")
user_norms = read_table("user_embedding_norms.csv")

for name, df in {
    "track_stats": track_stats,
    "track_norms": track_norms,
    "user_stats": user_stats,
    "user_norms": user_norms,
}.items():
    print(f"\n{name}: {df.shape}")
    show_df(df)


## Zero Vector And Norm Summary


In [ ]:
if not track_stats.empty:
    cols = ["column", "rows", "non_null", "nulls", "zero_vectors", "primary_dim", "norm_mean", "norm_median", "norm_min", "norm_max"]
    show_df(track_stats[[c for c in cols if c in track_stats.columns]].sort_values("zero_vectors", ascending=False), 50)

if not user_stats.empty:
    cols = ["split", "column", "rows", "non_null", "nulls", "zero_vectors", "primary_dim", "norm_mean", "norm_median", "norm_min", "norm_max"]
    show_df(user_stats[[c for c in cols if c in user_stats.columns]].sort_values(["split", "zero_vectors"], ascending=[True, False]), 50)

show_image("embedding_norms.png")


## Norm Distribution Checks


In [ ]:
if not track_norms.empty:
    sample = track_norms.copy()
    fig, ax = plt.subplots(figsize=(12, 4))
    if sns is not None:
        sns.histplot(data=sample, x="norm", hue="column", bins=60, element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title("Track embedding norm distribution")
    fig.tight_layout()
    plt.show()

if not user_norms.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    if sns is not None:
        sns.histplot(data=user_norms, x="norm", hue="split", bins=60, element="step", stat="density", common_norm=False, ax=ax)
    ax.set_title("User embedding norm distribution")
    fig.tight_layout()
    plt.show()


## Retrieval/Rerank Policy Buckets


In [ ]:
embedding_policy = pd.DataFrame([
    {"bucket": "track_zero_vector", "rule": "track embedding norm == 0", "diagnostic_use": "separate recall/nDCG metrics; dense retrieval cannot help directly"},
    {"bucket": "track_nonzero_vector", "rule": "track embedding norm > 0", "diagnostic_use": "eligible for dense candidate and cross-modal rerank"},
    {"bucket": "user_test_cold", "rule": "user cf-bpr vector is zero/missing", "diagnostic_use": "avoid user-CF source; prefer lexical/profile/history fallback"},
    {"bucket": "user_test_warm", "rule": "user cf-bpr vector is non-zero", "diagnostic_use": "test user-CF as feature-only before source mixing"},
])
show_df(embedding_policy, 20)


## Findings / Decisions / Next Actions

- Findings:
- Decisions:
- Next actions:
